In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
import pandas as pd
import numpy as np

In [ ]:
# =========================
# CONFIG
# =========================
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

TRAIN_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train"
TEST_DIR  = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final"

In [ ]:
# =========================
# DATA (WITH VALIDATION SPLIT)
# =========================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

In [ ]:
from tensorflow.keras import layers, models, regularizers

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# =========================
# CALLBACKS
# =========================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3
    )
]

In [ ]:
# # =========================
# # TRAIN
# # =========================
model.fit(
     train_data,
     validation_data=val_data,
     epochs=10,
     callbacks=callbacks
 )

In [ ]:
# =========================
# TEST DATA (FIXED)
# =========================
import os
import pandas as pd

test_datagen = ImageDataGenerator(rescale=1./255)

# get all test image filenames
files = os.listdir(TEST_DIR)

# create dataframe
df_test = pd.DataFrame({
    "filename": files
})

# load using dataframe (works for flat folders)
test_data = test_datagen.flow_from_dataframe(
    df_test,
    directory=TEST_DIR,
    x_col="filename",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode=None,
    shuffle=False
)


In [ ]:
# =========================
# PREDICT
# =========================
preds = model.predict(test_data).flatten()
labels = (preds > 0.5).astype(int)

In [ ]:
# map labels correctly
# IMPORTANT: check class indices
print(train_data.class_indices)

In [ ]:
# cell to be removed if labels are simply 0/1
# create reverse mapping 
idx_to_class = {v: k for k, v in train_data.class_indices.items()}

# convert predictions to string labels
string_labels = [idx_to_class[x] for x in labels]

In [ ]:
# create dataframe
df = pd.DataFrame({
    "ID": df_test["filename"],
    "Label": string_labels # replace string_labels with labels in case of 0/1
})

df.to_csv("submission.csv", index=False)

print("submission.csv created")